# `models.py` Reference

Core frozen dataclasses: `Player`, `Game`, `Match`, `Round`, `Tournament`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

ready


## Player

In [2]:
from tournament.models import AGENTS_TO_WIN, Player, Game, Match, Round, Tournament

print(f'AGENTS_TO_WIN = {AGENTS_TO_WIN}')

alice = Player(pid=1, name='Alice', skill=1.5)
bob   = Player(pid=2, name='Bob',   skill=0.5)
print(alice)
print(bob)

AGENTS_TO_WIN = 3
Player(pid=1, name='Alice', skill=1.5)
Player(pid=2, name='Bob', skill=0.5)


## Game

In [3]:
# Valid games
g_win  = Game(agents_a=3, agents_b=2)   # A wins
g_loss = Game(agents_a=1, agents_b=3)   # B wins
for g in (g_win, g_loss):
    valid, msg = g.is_valid
    print(f'Game({g.agents_a},{g.agents_b}): valid={valid}  winner_is_a={g.winner_is_a}')

# Invalid game — winner_is_a raises ValueError (guarded by check_validity)
g_bad = Game(agents_a=3, agents_b=3)
valid, msg = g_bad.is_valid
print(f'Game(3,3): valid={valid}  msg={msg!r}')
try:
    _ = g_bad.winner_is_a
except ValueError as e:
    print(f'winner_is_a raises: {e}')

Game(3,2): valid=True  winner_is_a=True
Game(1,3): valid=True  winner_is_a=False
Game(3,3): valid=False  msg='Both players have the same number of agents.'
winner_is_a raises: Both players have the same number of agents.


## Match

In [4]:
# Alice wins 2-0
m_win = Match(player_a=1, player_b=2, games=[Game(3, 1), Game(3, 2)])
print(f'2-0 match:  winner={m_win.winner}  is_draw={m_win.is_draw}')
print(f'  game_1_winner={m_win.game_1_winner}  game_2_winner={m_win.game_2_winner}')
print(f'  agent_score={m_win.agent_score}')
print(f'  total_agents_a={m_win.total_agents_a}  total_agents_b={m_win.total_agents_b}')
print(f'  results[1]={m_win.results[1]}')
print(f'  results[2]={m_win.results[2]}')

# Draw (1-1) with bonus tie-break
m_draw = Match(player_a=1, player_b=2,
               games=[Game(3, 2), Game(2, 3)],
               bonus_agents_a=1)
print(f'\nDraw match: winner={m_draw.winner}  is_draw={m_draw.is_draw}')
print(f'  total_agents_a={m_draw.total_agents_a}  agent_score={m_draw.agent_score}')

2-0 match:  winner=1  is_draw=False
  game_1_winner=1  game_2_winner=1
  agent_score=(6, 3)
  total_agents_a=6  total_agents_b=3
  results[1]={'id': 1, 'wins': 2, 'losses': 0, 'player_agents': 6, 'opponent_agents': 3, 'bonus_agents': 0, 'opponent': 2}
  results[2]={'id': 2, 'wins': 0, 'losses': 2, 'player_agents': 3, 'opponent_agents': 6, 'bonus_agents': 0, 'opponent': 1}

Draw match: winner=None  is_draw=True
  total_agents_a=6  agent_score=(6, 5)


## Round

In [5]:
rnd = Round(number=1, matches=[m_win])
valid, msg = rnd.is_valid
print(f'Round valid: {valid}')
print(f'opponents:   {rnd.opponents()}')
for player_id in rnd.players:
    print(f"result for player {player_id}: {rnd.player_results(player_id)}")


Round valid: True
opponents:   {1: 2, 2: 1}
result for player 1: {'id': 1, 'wins': 2, 'losses': 0, 'player_agents': 6, 'opponent_agents': 3, 'bonus_agents': 0, 'opponent': 2}
result for player 2: {'id': 2, 'wins': 0, 'losses': 2, 'player_agents': 3, 'opponent_agents': 6, 'bonus_agents': 0, 'opponent': 1}


## Tournament

In [6]:
tour = Tournament(players=[alice, bob])
tour.rounds.append(rnd)
print(f'player_ids: {tour.player_ids}')
print(f'player_by_id(1): {tour.player_by_id(1)}')
print(f'past_opponents(1): {tour.past_opponents(1)}')

# Add a second round
rnd2 = Round(number=2, matches=[
    Match(player_a=1, player_b=2, games=[Game(2, 3), Game(3, 2)])
])
tour.rounds.append(rnd2)
print(f'past_opponents(1) after round 2: {tour.past_opponents(1)}')
print(f'overall_results for round 2: {rnd2.overall_results}')

player_ids: [1, 2]
player_by_id(1): Player(pid=1, name='Alice', skill=1.5)
past_opponents(1): {2}
past_opponents(1) after round 2: {2}
overall_results for round 2: {'round': 2, 'matches': {1: {'id': 1, 'wins': 1, 'losses': 1, 'player_agents': 5, 'opponent_agents': 5, 'bonus_agents': 0, 'opponent': 2}, 2: {'id': 2, 'wins': 1, 'losses': 1, 'player_agents': 5, 'opponent_agents': 5, 'bonus_agents': 0, 'opponent': 1}}}
